# CNN sobre CIFAR-10 (imágenes en color) · Tema 2 · Sesión 3

**Máster en Ingeniería de Automatización con IA Agéntica · EBIS**
Autor: Manuel Díaz Bendito

MNIST es un problema fácil: dígitos en blanco y negro, centrados y limpios. **CIFAR-10** sube el listón: 60.000 fotos **en color** de 32×32 píxeles repartidas en 10 categorías reales —avión, coche, pájaro, gato, ciervo, perro, rana, caballo, barco, camión—. Hay fondos, iluminación variable, distintos ángulos... Aquí una CNN sencilla ya no llega al 99 %: el reto es realista.

Construimos una CNN en **PyTorch** con bloques convolucionales, BatchNorm, Dropout y data augmentation.

## 0 · Compatibilidad de hardware (léeme primero)

Este notebook funciona en **cualquier ordenador**. El código detecta automáticamente el mejor dispositivo de cómputo disponible y se adapta a él:

- **CUDA** → GPU NVIDIA (Windows / Linux). El más rápido para entrenar.
- **MPS** → GPUs de Apple Silicon (M1/M2/M3/M4). Acelera mucho en Mac.
- **CPU** → cualquier máquina. Funciona siempre; más lento, pero perfectamente válido para los modelos de este notebook.

No tienes que cambiar nada manualmente: la celda de detección de dispositivo elige sola. Si entrenas en CPU, los tiempos de este notebook siguen siendo de pocos minutos.

> **Nota sobre tiempos:** CIFAR-10 es más pesado que MNIST. En GPU (CUDA/MPS) cada época tarda segundos; en **CPU** puede tardar varios minutos por época. Si entrenas en CPU, baja `num_epocas` a 5-8 para una primera pasada: verás la dinámica de aprendizaje igualmente.

## 1 · Importación de librerías

In [ ]:
# %pip install torch torchvision numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42); np.random.seed(42)
print("PyTorch:", torch.__version__)

## 2 · Detección automática de dispositivo

In [ ]:
def elegir_dispositivo():
    """Devuelve el mejor dispositivo disponible: CUDA > MPS > CPU."""
    if torch.cuda.is_available():
        print(f"✅ Usando CUDA (GPU NVIDIA): {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        print("✅ Usando MPS (Apple Silicon)")
        return torch.device("mps")
    print("⚠️  No se ha detectado GPU. Usando CPU (funciona igual, algo más lento).")
    return torch.device("cpu")

device = elegir_dispositivo()
print("Dispositivo seleccionado:", device)

## 3 · Carga y preparación de CIFAR-10

Normalizamos cada canal de color (R, G, B) con su media y desviación, y aplicamos **data augmentation** al train (recorte aleatorio con padding y volteo horizontal), técnicas estándar que mejoran mucho la generalización en imágenes naturales.

In [ ]:
CLASES = ['avión','coche','pájaro','gato','ciervo','perro','rana','caballo','barco','camión']

media = (0.4914, 0.4822, 0.4465); desv = (0.2470, 0.2435, 0.2616)
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(media, desv),
])
transform_eval = transforms.Compose([transforms.ToTensor(), transforms.Normalize(media, desv)])

train_ds = datasets.CIFAR10(root="./datos", train=True,  download=True, transform=transform_train)
test_ds  = datasets.CIFAR10(root="./datos", train=False, download=True, transform=transform_eval)

batch_size = 128
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)
print(f"Train: {len(train_ds)} | Test: {len(test_ds)} | Clases: {CLASES}")

In [ ]:
# Visualizamos ejemplos (deshaciendo la normalización para verlos bien)
raw = datasets.CIFAR10(root="./datos", train=True, download=True, transform=transforms.ToTensor())
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax in axes.flatten():
    i = np.random.randint(len(raw)); img, lab = raw[i]
    ax.imshow(img.permute(1, 2, 0).numpy()); ax.set_title(CLASES[lab], fontsize=8); ax.axis("off")
plt.suptitle("Ejemplos de CIFAR-10"); plt.tight_layout(); plt.show()

## 4 · Arquitectura

Tres bloques convolucionales con profundidad creciente (32 → 64 → 128 filtros), cada uno con BatchNorm, doble convolución, pooling y Dropout. Es un patrón tipo *VGG reducido*, buen equilibrio entre rendimiento y coste en un portátil.

In [ ]:
class CNN_CIFAR(nn.Module):
    def __init__(self, n_clases=10):
        super().__init__()
        def bloque(ent, sal):
            return nn.Sequential(
                nn.Conv2d(ent, sal, 3, padding=1), nn.BatchNorm2d(sal), nn.ReLU(),
                nn.Conv2d(sal, sal, 3, padding=1), nn.BatchNorm2d(sal), nn.ReLU(),
                nn.MaxPool2d(2), nn.Dropout(0.25),
            )
        self.features = nn.Sequential(
            bloque(3, 32),    # 32x32 -> 16x16
            bloque(32, 64),   # 16x16 -> 8x8
            bloque(64, 128),  # 8x8 -> 4x4
        )
        self.clasificador = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128*4*4, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, n_clases),
        )
    def forward(self, x):
        return self.clasificador(self.features(x))

modelo = CNN_CIFAR().to(device)
print("Parámetros:", f"{sum(p.numel() for p in modelo.parameters()):,}")

## 5 · Entrenamiento

In [ ]:
import os
os.makedirs("./pesos", exist_ok=True)
RUTA_PESOS = "./pesos/cnn_cifar10.pt"

def evaluar(modelo, loader):
    modelo.eval(); correctos = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correctos += (modelo(x).argmax(1) == y).sum().item(); total += y.size(0)
    return correctos / total

criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=20)

num_epocas = 20          # baja a 5-8 si entrenas en CPU
mejor = 0.0
hist = {"train_loss": [], "test_acc": []}
for epoca in range(num_epocas):
    modelo.train(); perdida = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        loss = criterio(modelo(x), y)
        optimizador.zero_grad(); loss.backward(); optimizador.step()
        perdida += loss.item()
    scheduler.step()
    acc = evaluar(modelo, test_loader)
    hist["train_loss"].append(perdida/len(train_loader)); hist["test_acc"].append(acc)
    if acc > mejor:
        mejor = acc; torch.save(modelo.state_dict(), RUTA_PESOS)
    print(f"Época {epoca+1:2d}/{num_epocas} | loss {hist['train_loss'][-1]:.4f} | test acc {acc*100:.2f} %")
print(f"\nMejor accuracy en test: {mejor*100:.2f} %")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
a1.plot(hist["train_loss"], color="#08627F"); a1.set_title("Loss de entrenamiento"); a1.set_xlabel("Época"); a1.grid(alpha=0.3)
a2.plot([v*100 for v in hist["test_acc"]], color="#6ACBB8"); a2.set_title("Accuracy en test"); a2.set_xlabel("Época"); a2.set_ylabel("%"); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6 · Evaluación final y análisis por clase

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

modelo.load_state_dict(torch.load(RUTA_PESOS, map_location=device))
modelo.eval(); y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        y_pred.extend(modelo(x.to(device)).argmax(1).cpu().tolist()); y_true.extend(y.tolist())

print(f"Accuracy en test: {np.mean(np.array(y_true)==np.array(y_pred))*100:.2f} %\n")
print(classification_report(y_true, y_pred, target_names=CLASES, digits=3))
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(cmap="Blues", ax=ax, colorbar=False, xticks_rotation=45)
plt.title("Matriz de confusión · CIFAR-10"); plt.tight_layout(); plt.show()

## 7 · Predicciones de ejemplo

In [ ]:
raw_test = datasets.CIFAR10(root="./datos", train=False, download=True, transform=transforms.ToTensor())
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax in axes.flatten():
    i = np.random.randint(len(test_ds))
    x, real = test_ds[i]
    with torch.no_grad():
        pred = modelo(x.unsqueeze(0).to(device)).argmax(1).item()
    ax.imshow(raw_test[i][0].permute(1,2,0).numpy())
    color = "green" if pred == real else "red"
    ax.set_title(f"{CLASES[real]}→{CLASES[pred]}", color=color, fontsize=8); ax.axis("off")
plt.suptitle("Predicciones (verde = acierto, rojo = fallo)"); plt.tight_layout(); plt.show()

## 8 · Lo que has aprendido

- CIFAR-10 muestra por qué las imágenes **reales en color** son mucho más difíciles que MNIST.
- Una CNN profunda con **BatchNorm, Dropout, data augmentation** y un buen **scheduler** alcanza un rendimiento sólido (~85-88 %) en un portátil.
- La matriz de confusión revela errores **semánticamente razonables** (gato↔perro, coche↔camión): la red confunde clases visualmente parecidas, igual que haría una persona con miniaturas de 32×32.

> Para superar este techo se usan arquitecturas más potentes (ResNet, etc.) y *transfer learning*, que veréis aplicado en temas posteriores.